In [19]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path


notebook_dir = Path(os.getcwd())
root_dir = notebook_dir.parents[0] 
env_path = root_dir / ".env"

# Charge le fichier .env
load_dotenv(dotenv_path=env_path)

EIA_API_KEY = os.getenv("EIA_API_KEY")
print("Chemin du .env cherché :", env_path.resolve())
print("Ma clé API chargée :", EIA_API_KEY)

Chemin du .env cherché : C:\Users\user\Desktop\Gasoil_Intelligence\plugins\scripts\.env
Ma clé API chargée : Ywlv9ChPz0AzigFBT9lGWsrRjwyOafGJlYD9QLvE


In [20]:
# Requête sur l'API EIA pour récupérer les données de vente de Gasoil
url = f"https://api.eia.gov/v2/petroleum/pri/gnd/data/?api_key={EIA_API_KEY}&frequency=weekly&data[0]=value&sort[0][column]=period&sort[0][direction]=desc&length=100"

response = requests.get(url).json()
records = response['response']['data']

# On crée enfin la variable df_raw ici !
df_raw = pd.DataFrame(records)
df_raw.info()
df_raw.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   period              100 non-null    object
 1   duoarea             100 non-null    object
 2   area-name           100 non-null    object
 3   product             100 non-null    object
 4   product-name        100 non-null    object
 5   process             100 non-null    object
 6   process-name        100 non-null    object
 7   series              100 non-null    object
 8   series-description  100 non-null    object
 9   value               99 non-null     object
 10  units               100 non-null    object
dtypes: object(11)
memory usage: 8.7+ KB


,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2026-07-13,NUS,U.S.,EPD2DXL0,No 2 Diesel Low Sulfur (0-15 ppm),PTE,Retail Sales,EMD_EPD2DXL0_PTE_NUS_DPG,U.S. No 2 Diesel Ultra Low Sulfur (0-15 ppm) R...,4.796,$/GAL
1,2026-07-13,R20,PADD 2,EPMMR,Gasoline Reformulated Midgrade,PTE,Retail Sales,EMM_EPMMR_PTE_R20_DPG,Midwest Midgrade Reformulated Retail Gasoline ...,4.45,$/GAL
2,2026-07-13,R1Z,PADD 1C,EPMMR,Gasoline Reformulated Midgrade,PTE,Retail Sales,EMM_EPMMR_PTE_R1Z_DPG,Lower Atlantic (PADD 1C) Midgrade Reformulated...,4.235,$/GAL
3,2026-07-13,R1Y,PADD 1B,EPMMR,Gasoline Reformulated Midgrade,PTE,Retail Sales,EMM_EPMMR_PTE_R1Y_DPG,Central Atlantic (PADD 1B) Midgrade Reformulat...,4.49,$/GAL
4,2026-07-13,R1X,PADD 1A,EPMMR,Gasoline Reformulated Midgrade,PTE,Retail Sales,EMM_EPMMR_PTE_R1X_DPG,New England (PADD 1A) Midgrade Reformulated Re...,4.478,$/GAL


In [21]:
df_clean=pd.DataFrame()
df_clean['date']=pd.to_datetime(df_raw['period']).dt.date
df_clean['product_type']=df_raw['product-name'].astype(str)
df_clean['region']=df_raw['area-name'].astype(str)
df_clean['price_usd_per_gallon']=pd.to_numeric(df_raw['value'])
if 'process-desc' in df_raw.columns:
    df_clean['grade'] = df_raw['process-desc'].astype(str)
elif 'process_desc' in df_raw.columns:
    df_clean['grade'] = df_raw['process_desc'].astype(str)
else:
    df_clean['grade'] = 'Standard'

df_clean['source']='EIA'
df_clean=df_clean.dropna(subset=['price_usd_per_gallon'])
df_clean=df_clean[df_clean['price_usd_per_gallon']>0]
#supression des doublons
df_clean=df_clean.drop_duplicates(subset=['date','product_type','region','grade','source'])
df_clean.head()

,date,product_type,region,price_usd_per_gallon,grade,source
0,2026-07-13,No 2 Diesel Low Sulfur (0-15 ppm),U.S.,4.796,Standard,EIA
1,2026-07-13,Gasoline Reformulated Midgrade,PADD 2,4.450,Standard,EIA
2,2026-07-13,Gasoline Reformulated Midgrade,PADD 1C,4.235,Standard,EIA
3,2026-07-13,Gasoline Reformulated Midgrade,PADD 1B,4.490,Standard,EIA
4,2026-07-13,Gasoline Reformulated Midgrade,PADD 1A,4.478,Standard,EIA


In [22]:
from sqlalchemy import create_engine

DB_USER = "airflow"
DB_PASSWORD = "airflow"  
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "airflow"  

connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

try:
    
    df_clean.to_sql(
        name='gasoil_prices', 
        con=engine, 
        if_exists='append', 
        index=False
    )
    print("Succès ! Les données nettoyées ont été correctement insérées dans la table 'gasoil_prices'.")
except Exception as e:
    print("Erreur lors de l'insertion dans la base de données :")
    print(e)

Succès ! Les données nettoyées ont été correctement insérées dans la table 'gasoil_prices'.
